# FedSGD and FedAvg on IID Data

Implements federated averaging inline and measures client participation fraction C and local computation E and B.

In [1]:
import os
import copy
import pandas as pd
import torch
import torch.nn as nn
from src.utils import seed_everything, load_mnist, evaluate
from src.models import TwoNN
from src.partition import load_partition

seed_everything(42)
os.makedirs('results', exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
(train_x, train_y), (test_x, test_y) = load_mnist('data')
partition = load_partition('data/iid_k100.npz')
client_data = {c: (train_x[idx], train_y[idx]) for c, idx in partition.items()}

Inline definition of client update and FedAvg round.

In [2]:
def client_update(model, x, y, epochs=1, batch_size=10, lr=0.01, device=None):
    if device is None:
        device = next(model.parameters()).device
    local_model = copy.deepcopy(model).to(device)
    local_model.train()
    optimizer = torch.optim.SGD(local_model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    num_samples = len(y)
    bs = num_samples if (batch_size is None or batch_size >= num_samples) else batch_size
    for _ in range(epochs):
        perm = torch.randperm(num_samples)
        for i in range(0, num_samples, bs):
            b_idx = perm[i:i + bs]
            bx, by = x[b_idx].to(device), y[b_idx].to(device)
            optimizer.zero_grad()
            out = local_model(bx)
            loss = criterion(out, by)
            loss.backward()
            optimizer.step()
    return local_model.state_dict()

def fedavg_round(global_model, client_data, selected_clients, epochs=1, batch_size=10, lr=0.01, device=None):
    if device is None:
        device = next(global_model.parameters()).device
    total_samples = sum(len(client_data[c][1]) for c in selected_clients)
    global_dict = global_model.state_dict()
    new_dict = copy.deepcopy(global_dict)
    for key in new_dict:
        new_dict[key] = torch.zeros_like(new_dict[key], dtype=torch.float32)
    for c in selected_clients:
        cx, cy = client_data[c]
        c_weights = client_update(global_model, cx, cy, epochs=epochs, batch_size=batch_size, lr=lr, device=device)
        weight = len(cy) / total_samples
        for key in new_dict:
            new_dict[key] += c_weights[key] * weight
    global_model.load_state_dict(new_dict)

Experiment E3a: vary client fraction C in FedSGD (E=1, B=600).

In [3]:
c_records = []
c_values = [0.0, 0.1, 0.2, 0.5, 1.0]
rounds = 8
lr = 0.1

for c_frac in c_values:
    seed_everything(42)
    global_model = TwoNN().to(device)
    num_selected = max(int(c_frac * 100), 1)
    
    loss, acc = evaluate(global_model, test_x, test_y, device=device)
    c_records.append({'exp': 'E3a', 'model': '2NN', 'algo': 'FedSGD', 'partition': 'IID',
                      'K': 100, 'C': c_frac, 'E': 1, 'B': 600, 'lr': lr, 'seed': 42,
                      'round': 0, 'test_acc': acc, 'test_loss': loss})
    
    for rnd in range(1, rounds + 1):
        perm = torch.randperm(100)
        selected = perm[:num_selected].tolist()
        fedavg_round(global_model, client_data, selected, epochs=1, batch_size=600, lr=lr, device=device)
        loss, acc = evaluate(global_model, test_x, test_y, device=device)
        c_records.append({'exp': 'E3a', 'model': '2NN', 'algo': 'FedSGD', 'partition': 'IID',
                          'K': 100, 'C': c_frac, 'E': 1, 'B': 600, 'lr': lr, 'seed': 42,
                          'round': rnd, 'test_acc': acc, 'test_loss': loss})
    print(f'FedSGD C={c_frac} round {rounds}: test_acc={acc:.4f}')

pd.DataFrame(c_records).to_csv('results/fedsgd_participation.csv', index=False)

FedSGD C=0.0 round 8: test_acc=0.5655


FedSGD C=0.1 round 8: test_acc=0.6226


FedSGD C=0.2 round 8: test_acc=0.6195


FedSGD C=0.5 round 8: test_acc=0.6273


FedSGD C=1.0 round 8: test_acc=0.6228


Experiment E3b: vary local computation E and batch size B in FedAvg.

In [4]:
b_records = []
configs = [
    {'E': 1, 'B': 600, 'lr': 0.1},
    {'E': 1, 'B': 50, 'lr': 0.05},
    {'E': 5, 'B': 10, 'lr': 0.03}
]
rounds = 8
c_count = 10

for cfg in configs:
    seed_everything(42)
    global_model = TwoNN().to(device)
    e_val, b_val, lr_val = cfg['E'], cfg['B'], cfg['lr']
    
    loss, acc = evaluate(global_model, test_x, test_y, device=device)
    b_records.append({'exp': 'E3b', 'model': '2NN', 'algo': 'FedAvg', 'partition': 'IID',
                      'K': 100, 'C': 0.1, 'E': e_val, 'B': b_val, 'lr': lr_val, 'seed': 42,
                      'round': 0, 'test_acc': acc, 'test_loss': loss})
    
    for rnd in range(1, rounds + 1):
        perm = torch.randperm(100)
        selected = perm[:c_count].tolist()
        fedavg_round(global_model, client_data, selected, epochs=e_val, batch_size=b_val, lr=lr_val, device=device)
        loss, acc = evaluate(global_model, test_x, test_y, device=device)
        b_records.append({'exp': 'E3b', 'model': '2NN', 'algo': 'FedAvg', 'partition': 'IID',
                          'K': 100, 'C': 0.1, 'E': e_val, 'B': b_val, 'lr': lr_val, 'seed': 42,
                          'round': rnd, 'test_acc': acc, 'test_loss': loss})
    print(f'FedAvg E={e_val} B={b_val} round {rounds}: test_acc={acc:.4f}')

pd.DataFrame(b_records).to_csv('results/fedavg_computation.csv', index=False)

FedAvg E=1 B=600 round 8: test_acc=0.6226


FedAvg E=1 B=50 round 8: test_acc=0.8764


FedAvg E=5 B=10 round 8: test_acc=0.9474


More local updates per round (higher E, smaller B) substantially reduce the number of rounds required to attain high accuracy.